<div style="text-align:center;">

# Modelo: XGBoost Classifier
### *Clasificación de la Rotación de Empleados (IBM HR Analytics)*

---
**Universidad de Antioquia — Instituto de Matemáticas**  
*Analítica de Datos — Laboratorio 5*

</div>

---

## Contenido de este notebook

Este notebook desarrolla — de forma autocontenida — los siguientes puntos del Laboratorio 5 para el algoritmo **XGBoost**:

3. **Preparación de los Datos** — separación X / y, partición train/test estratificada, construcción del `Pipeline` con codificación de variables categóricas y escalamiento.
4. **Entrenamiento del modelo** dentro del `Pipeline`.
5. **Validación cruzada estratificada** (`StratifiedKFold`, k = 5) con métricas reportadas como $\mu \pm \sigma$ y visualización del comportamiento entre folds.

> Las decisiones de preprocesamiento fueron tomadas en el notebook `0.Exploracion_Inicial.ipynb` y aquí se replican estrictamente, garantizando comparabilidad con los demás modelos.

### Descripción del algoritmo

**XGBoost** (Extreme Gradient Boosting) construye un ensamble **secuencial** de árboles donde cada nuevo árbol corrige los errores residuales del ensamble anterior, optimizando la pérdida mediante **gradient boosting** con regularización L1/L2. Es uno de los algoritmos **más competitivos** en problemas tabulares y suele ganar la comparación contra Random Forest en datasets de tamaño medio.

### Tratamiento del desbalance

XGBoost no acepta `class_weight`. Su mecanismo equivalente es `scale_pos_weight = n_neg / n_pos`, que escala el gradiente de la clase positiva en la función de pérdida — el resultado es matemáticamente análogo a reponderar la entropía cruzada.


---
## 1. Importación de librerías y configuración

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

# Preprocesamiento y pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelo
from xgboost import XGBClassifier

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint

# Validación cruzada y métricas
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, cross_val_predict)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,fbeta_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

# ── Estilo visual unificado ──────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi"        : 120,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.titleweight"  : "bold",
    "axes.titlesize"    : 12,
    "axes.labelsize"    : 10,
    "font.size"         : 10,
    "legend.frameon"    : False,
})

COLOR_YES, COLOR_NO = "#e74c3c", "#3498db"
COLOR_MODELO = "#2c3e50"      # color principal del modelo

print("Librerías cargadas correctamente.")

---
## 2. Carga y preparación del conjunto de datos

In [ ]:
DATA_PATH = Path.cwd().parent.parent / "data" / "raw" / "dataset_clasificacion.csv"

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

# ── Eliminamos columnas sin información (constantes / identificador) ─────
COLS_DESCARTAR = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]
df = df.drop(columns=COLS_DESCARTAR)

print(f"Dimensiones tras descartar columnas inútiles: {df.shape}")
df.head(3)

---
## 3. Preparación de los Datos

### 3.1 Separación de variables predictoras (X) y objetivo (y)

Se codifica el target de manera explícita: $\text{Yes} \mapsto 1,\ \text{No} \mapsto 0$, con la clase positiva siendo la **renuncia** (que es la que interesa detectar).


In [ ]:
y = (df["Attrition"] == "Yes").astype(int)
X = df.drop(columns=["Attrition"])

print(f"X : {X.shape[0]:,} observaciones × {X.shape[1]} features")
print(f"y : binaria — Yes(1) = {y.sum()}  |  No(0) = {(y==0).sum()}")
print(f"    Proporción de la clase positiva: {y.mean()*100:.2f} %")

### 3.2 Identificación del tipo de variables

Se replican las decisiones tomadas en `0.Exploracion_Inicial.ipynb`:

| Grupo | Tratamiento |
|:---|:---|
| **Nominales** (7) | One-Hot Encoding con `drop="first"` |
| **Numéricas + Ordinales** (23) | Estandarización con `StandardScaler` |


In [ ]:
COLS_NOMINALES = ["BusinessTravel", "Department", "EducationField",
                  "Gender", "JobRole", "MaritalStatus", "OverTime"]

COLS_NUMERICAS = [c for c in X.columns if c not in COLS_NOMINALES]

print(f"Nominales ({len(COLS_NOMINALES)}) : {COLS_NOMINALES}")
print(f"\nNuméricas ({len(COLS_NUMERICAS)}) : {COLS_NUMERICAS}")

### 3.3 Partición Train / Test estratificada

- Proporción **70 / 30** (entrenamiento / prueba).
- Se usa **estratificación por la variable objetivo** (`stratify=y`) para preservar la proporción $16.1\,/\,83.9$ en ambos subconjuntos.
- `random_state=42` para reproducibilidad.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]:,} obs.  | Yes = {y_train.mean()*100:.2f} %")
print(f"Test  : {X_test.shape[0]:,} obs.  | Yes = {y_test.mean()*100:.2f} %")

### 3.4 Construcción del Pipeline de preprocesamiento

El `ColumnTransformer` aplica **One-Hot Encoding** a las nominales y **estandarización** a las numéricas. Al estar **dentro del `Pipeline`**, ambas transformaciones son ajustadas únicamente sobre cada fold de entrenamiento durante la validación cruzada, evitando cualquier **fuga de información** desde los folds de validación.

> **Importante**: tanto la codificación como el escalado deben aprenderse exclusivamente sobre los datos de entrenamiento de cada fold. El uso del `Pipeline` garantiza esta propiedad de manera automática.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), COLS_NOMINALES),
        ("num", StandardScaler(), COLS_NUMERICAS),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
preprocessor

---
## 4. Construcción y entrenamiento del modelo

### 4.1 Instanciación dentro del `Pipeline`

El modelo se integra como segundo paso del `Pipeline`. Toda evaluación posterior se hace sobre el `Pipeline` completo (preprocesamiento + clasificador), de modo que el escalado y la codificación se ajustan **en cada fold** sobre los datos de entrenamiento únicamente.


In [ ]:
# scale_pos_weight equivale a class_weight="balanced" para XGBoost
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
print(f"scale_pos_weight = n_neg / n_pos = {n_neg} / {n_pos} = {n_neg/n_pos:.4f}")

modelo = Pipeline(steps=[
    ("preprocesamiento", preprocessor),
    ("clf", XGBClassifier(scale_pos_weight=n_neg/n_pos, n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42, eval_metric="logloss", verbosity=0)),
])
modelo

### 4.2 Entrenamiento sobre el conjunto de entrenamiento completo

In [ ]:
modelo.fit(X_train, y_train)
print("Modelo entrenado correctamente.")

---
## 5. Validación Cruzada Estratificada (k = 5)

### 5.1 Diseño del experimento

- **`StratifiedKFold`** con `n_splits=5`, `shuffle=True`, `random_state=42` — preserva la proporción de clases en cada fold.
- Se evalúan **cinco métricas** simultáneamente: `accuracy`, `precision`, `recall`, `f1` y `roc_auc`.
- El `Pipeline` completo se reajusta en cada fold ⇒ no hay fuga de información del preprocesamiento.
- Los resultados se reportan como $\mu \pm \sigma$ sobre los 5 folds.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy" : "accuracy",
    "precision": "precision",
    "recall"   : "recall",
    "f1"       : "f1",
    "roc_auc"  : "roc_auc",
}

cv_results = cross_validate(
    modelo, X_train, y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Validación cruzada finalizada.")
print(f"Número de folds: {skf.get_n_splits()}")

### 5.2 Resultados consolidados — $\mu \pm \sigma$

In [ ]:
def resumen_cv(cv_results, metricas):
    filas = []
    for m in metricas:
        train = cv_results[f"train_{m}"]
        test  = cv_results[f"test_{m}"]
        filas.append({
            "Métrica"          : m.upper(),
            "Train  μ"         : train.mean(),
            "Train  σ"         : train.std(),
            "Validación  μ"    : test.mean(),
            "Validación  σ"    : test.std(),
            "Gap (Train − Val)": train.mean() - test.mean(),
        })
    return pd.DataFrame(filas).set_index("Métrica").round(4)

tabla_cv = resumen_cv(cv_results, ["accuracy", "precision", "recall", "f1", "roc_auc"])
tabla_cv

In [ ]:
# Reporte en formato μ ± σ (limpio para presentación)
print("─" * 56)
print(f"  {'Métrica':<12} {'Validación (μ ± σ)':>22}  {'Train (μ ± σ)':>18}")
print("─" * 56)
for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    v = cv_results[f"test_{m}"]
    t = cv_results[f"train_{m}"]
    print(f"  {m.upper():<12} {v.mean():>8.4f} ± {v.std():.4f}     {t.mean():>6.4f} ± {t.std():.4f}")
print("─" * 56)

### 5.3 Visualización 1 — Distribución de métricas por fold

In [ ]:
metricas = ["accuracy", "precision", "recall", "f1", "roc_auc"]
nombres  = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC"]

# DataFrame "tidy" para seaborn
data_tidy = []
for m, nom in zip(metricas, nombres):
    for fold, val in enumerate(cv_results[f"test_{m}"], start=1):
        data_tidy.append({"Métrica": nom, "Fold": fold, "Valor": val})
df_tidy = pd.DataFrame(data_tidy)

fig, ax = plt.subplots(figsize=(13, 6))

sns.boxplot(data=df_tidy, x="Métrica", y="Valor",
            color=COLOR_MODELO, width=0.45, fliersize=0, ax=ax, linewidth=1.4,
            boxprops=dict(alpha=0.35))
sns.stripplot(data=df_tidy, x="Métrica", y="Valor",
              color=COLOR_MODELO, size=7, jitter=0.12, alpha=0.85,
              edgecolor="white", linewidth=0.8, ax=ax)

# Anotación de la media en cada métrica
for i, m in enumerate(metricas):
    media = cv_results[f"test_{m}"].mean()
    std   = cv_results[f"test_{m}"].std()
    ax.text(i, media, f" Media = {media:.3f}\n Desv. = {std:.3f}",
            fontsize=9, ha="left", va="center",
            bbox=dict(boxstyle="round,pad=0.3",
                      facecolor="white", edgecolor=COLOR_MODELO, alpha=0.85))
    ax.scatter(i, media, color="red", s=80, zorder=5,
               marker="D", edgecolor="white", linewidth=1.2)

ax.set_ylim(0, 1.02)
ax.set_ylabel("Valor de la métrica (validación)")
ax.set_xlabel("")
ax.set_title("Distribución de las métricas a través de los 5 folds",
             fontsize=13, pad=12)
ax.grid(axis="y", ls="--", alpha=0.35)

ax.text(0.99, 1.02, "♦ media",
        transform=ax.transAxes, ha="right", fontsize=9, color="red")

plt.tight_layout()
plt.show()

### 5.4 Visualización 2 — Comparación Train vs. Validación (diagnóstico de sobreajuste)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(metricas))
ancho = 0.35

medias_train = [cv_results[f"train_{m}"].mean() for m in metricas]
stds_train   = [cv_results[f"train_{m}"].std()  for m in metricas]
medias_val   = [cv_results[f"test_{m}"].mean()  for m in metricas]
stds_val     = [cv_results[f"test_{m}"].std()   for m in metricas]

bars_t = ax.bar(x - ancho/2, medias_train, ancho, yerr=stds_train, capsize=4,
                color="#bdc3c7", edgecolor="white", label="Train",
                error_kw=dict(elinewidth=1.4, ecolor="#34495e"))
bars_v = ax.bar(x + ancho/2, medias_val, ancho, yerr=stds_val, capsize=4,
                color=COLOR_MODELO, edgecolor="white", label="Validación",
                error_kw=dict(elinewidth=1.4, ecolor="#1a1a1a"))

# Etiquetas numéricas
for b, val in zip(bars_t, medias_train):
    ax.text(b.get_x() + b.get_width()/2, val + 0.018,
            f"{val:.3f}", ha="center", fontsize=8.5, color="#34495e")
for b, val in zip(bars_v, medias_val):
    ax.text(b.get_x() + b.get_width()/2, val + 0.018,
            f"{val:.3f}", ha="center", fontsize=8.5, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(nombres)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Valor de la métrica")
ax.set_title("Comparativo Train vs. Validación — diagnóstico de sobreajuste",
             fontsize=13, pad=12)
ax.legend(loc="upper right", frameon=True, fancybox=True)
ax.grid(axis="y", ls="--", alpha=0.35)
plt.tight_layout()
plt.show()

### 5.5 Visualización 3 — Curva ROC promedio sobre los 5 folds

Promediamos las curvas ROC obtenidas en cada fold de validación e indicamos la franja $\pm 1\sigma$ del TPR para cuantificar la **estabilidad** del clasificador entre folds.


In [ ]:
mean_fpr = np.linspace(0, 1, 200)
tprs, aucs = [], []

fig, ax = plt.subplots(figsize=(8.5, 7.5))

for i, (idx_tr, idx_va) in enumerate(skf.split(X_train, y_train), start=1):
    Xtr, Xva = X_train.iloc[idx_tr], X_train.iloc[idx_va]
    ytr, yva = y_train.iloc[idx_tr], y_train.iloc[idx_va]

    modelo.fit(Xtr, ytr)
    y_score = modelo.predict_proba(Xva)[:, 1]
    fpr, tpr, _ = roc_curve(yva, y_score)
    auc_fold = roc_auc_score(yva, y_score)

    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)
    aucs.append(auc_fold)

    ax.plot(fpr, tpr, lw=1.2, alpha=0.45,
            label=f"Fold {i}  (AUC = {auc_fold:.3f})")

mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
std_tpr   = np.std(tprs, axis=0)
mean_auc  = np.mean(aucs)
std_auc   = np.std(aucs)

ax.plot(mean_fpr, mean_tpr, color=COLOR_MODELO, lw=2.5,
        label=f"ROC promedio  (AUC = {mean_auc:.3f} ± {std_auc:.3f})")
ax.fill_between(mean_fpr, np.maximum(mean_tpr - std_tpr, 0),
                np.minimum(mean_tpr + std_tpr, 1),
                color=COLOR_MODELO, alpha=0.15,
                label="± 1 σ entre folds")

ax.plot([0,1], [0,1], ls="--", color="gray", lw=1, label="Clasificador aleatorio")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Tasa de Falsos Positivos (1 − Especificidad)")
ax.set_ylabel("Tasa de Verdaderos Positivos (Sensibilidad)")
ax.set_title("Curva ROC — Promedio sobre 5 folds de validación", pad=12)
ax.legend(loc="lower right", fontsize=9, frameon=True, fancybox=True)
ax.grid(ls="--", alpha=0.35)
plt.tight_layout()
plt.show()

# Re-ajustamos el modelo sobre todo el train para dejarlo entrenado al final
modelo.fit(X_train, y_train);

### 5.6 Visualización 4 — Matriz de Confusión (predicciones out-of-fold)

La matriz se calcula a partir de **`cross_val_predict`**, que agrega las predicciones de cada observación del conjunto de entrenamiento **cuando ésta caía en el fold de validación**. De esta manera, cada observación se predice una sola vez con un modelo que **no la vio durante su entrenamiento** — la matriz refleja el desempeño esperado sobre datos no vistos.

> Es la misma información que entregan las métricas $\mu \pm \sigma$ de la sección 5.2, pero desagregada en los cuatro tipos de error/acierto del clasificador binario.


In [ ]:
from sklearn.metrics import confusion_matrix

# Predicciones out-of-fold sobre el conjunto de entrenamiento
y_pred_cv = cross_val_predict(modelo, X_train, y_train, cv=skf, n_jobs=-1)

cm = confusion_matrix(y_train, y_pred_cv, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

# Cálculos derivados — porcentajes por fila (sensibilidad por clase real)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100

# ── Figura: matriz en conteos y en porcentajes ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

etiquetas = ["No (permanece)", "Yes (renuncia)"]
descripcion = np.array([
    [f"Verdaderos\nNegativos\n(TN)",  f"Falsos\nPositivos\n(FP)"],
    [f"Falsos\nNegativos\n(FN)",       f"Verdaderos\nPositivos\n(TP)"],
])

# ── Panel A: conteos absolutos ───────────────────────────────────────────
ax = axes[0]
im = ax.imshow(cm, cmap="Blues", aspect="auto")
for i in range(2):
    for j in range(2):
        valor = cm[i, j]
        color = "white" if valor > cm.max() * 0.55 else "#1a1a1a"
        ax.text(j, i - 0.13, f"{valor:,}",
                ha="center", va="center",
                color=color, fontsize=18, fontweight="bold")
        ax.text(j, i + 0.22, descripcion[i, j],
                ha="center", va="center", color=color, fontsize=8.5)

ax.set_xticks([0, 1]); ax.set_xticklabels(etiquetas, fontsize=10)
ax.set_yticks([0, 1]); ax.set_yticklabels(etiquetas, fontsize=10, rotation=90, va="center")
ax.set_xlabel("Clase predicha", fontsize=11)
ax.set_ylabel("Clase real", fontsize=11)
ax.set_title("Conteos absolutos", fontsize=12, pad=10)

# ── Panel B: porcentajes por clase real (fila normalizada) ───────────────
ax = axes[1]
im = ax.imshow(cm_pct, cmap="Blues", aspect="auto", vmin=0, vmax=100)
for i in range(2):
    for j in range(2):
        valor = cm_pct[i, j]
        color = "white" if valor > 55 else "#1a1a1a"
        ax.text(j, i - 0.10, f"{valor:.1f} %",
                ha="center", va="center",
                color=color, fontsize=18, fontweight="bold")
        ax.text(j, i + 0.20, f"({cm[i, j]:,} / {cm[i].sum():,})",
                ha="center", va="center", color=color, fontsize=9)

ax.set_xticks([0, 1]); ax.set_xticklabels(etiquetas, fontsize=10)
ax.set_yticks([0, 1]); ax.set_yticklabels(etiquetas, fontsize=10, rotation=90, va="center")
ax.set_xlabel("Clase predicha", fontsize=11)
ax.set_ylabel("Clase real", fontsize=11)
ax.set_title("Porcentajes por clase real\n(fila normalizada)", fontsize=12, pad=10)

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("% por clase real", rotation=270, labelpad=15)

fig.suptitle("Matriz de Confusión — Predicciones Out-of-Fold (Validación Cruzada k = 5)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── Resumen numérico ─────────────────────────────────────────────────────
print("─" * 60)
print(f"  Verdaderos Negativos (TN) : {tn:>5,}    — correctamente identificados como 'No'")
print(f"  Falsos Positivos     (FP) : {fp:>5,}    — falsamente alertados como 'Yes'")
print(f"  Falsos Negativos     (FN) : {fn:>5,}    — renuncias NO detectadas")
print(f"  Verdaderos Positivos (TP) : {tp:>5,}    — renuncias correctamente detectadas")
print("─" * 60)
print(f"  Sensibilidad (Recall sobre Yes) : {tp/(tp+fn):.4f}")
print(f"  Especificidad (Recall sobre No) : {tn/(tn+fp):.4f}")
print(f"  Precisión (sobre Yes)           : {tp/(tp+fp) if (tp+fp)>0 else 0:.4f}")
print("─" * 60)

---
## 6. Persistencia de los resultados de CV

Guardamos los scores por fold y las medias en variables `cv_metrics_<modelo>` para reutilizarlos en el **notebook de comparación final** (`5.Comparacion_Modelos.ipynb`).


In [ ]:
cv_metrics = {
    "modelo"  : "XGBoost",
    "metricas": {
        m: {
            "scores": cv_results[f"test_{m}"].tolist(),
            "mu"    : float(cv_results[f"test_{m}"].mean()),
            "sigma" : float(cv_results[f"test_{m}"].std()),
        }
        for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]
    }
}

# Guardamos en JSON para que el notebook de comparación los lea
import json, os
os.makedirs("cv_results", exist_ok=True)
out_path = os.path.join("cv_results", "xgboost.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(cv_metrics, f, indent=2, ensure_ascii=False)

print(f"Resultados de CV guardados en: {out_path}")

---
## 7. Síntesis del modelo XGBoost

A partir de la validación cruzada $k=5$ obtenemos un perfil **estable** del comportamiento esperado del modelo:

- Las **cinco métricas** se reportan como $\mu \pm \sigma$ sobre los folds — la columna `Validación` en la tabla 5.2 es la lectura honesta del desempeño esperado en datos nuevos.
- El **gap Train − Validación** (última columna) cuantifica el **sobreajuste**: valores cercanos a cero indican buena generalización; valores positivos grandes (>0.05–0.10) sugieren que el modelo memoriza el entrenamiento.
- La **curva ROC promedio** con su franja $\pm 1\sigma$ resume la capacidad discriminativa global.

> El análisis comparativo entre los cuatro modelos y la elección del mejor se realiza en `5.Comparacion_Modelos.ipynb`.


In [ ]:
# Parámetros distribuidos para XGBoost (Alineado con el Notebook)
xgb_param_dist = {
    'clf__n_estimators': randint(50, 400),       # Número de árboles entre 50 y 400
    'clf__max_depth': randint(3, 10),            # Profundidad de los árboles entre 3 y 10
    'clf__learning_rate': loguniform(0.005, 0.3), # Tasa de aprendizaje en escala logarítmica
    'clf__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],  # Porcentaje de filas por árbol
    'clf__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0], # Porcentaje de columnas por árbol
    'clf__scale_pos_weight': [1, 2, 3, 4, 5]     # Control de desbalance (peso clase positiva)
}

# Configuración del RandomizedSearchCV
xgb_search = RandomizedSearchCV(
    estimator=modelo, 
    param_distributions=xgb_param_dist, 
    n_iter=20, 
    cv=5, 
    scoring='roc_auc', 
    random_state=42, 
    n_jobs=-1
)

# Ajustar el modelo
xgb_search.fit(X_train, y_train)

print("Mejores parámetros para XGBoost:")
print(xgb_search.best_params_)

In [ ]:
# 1. Extraer métricas individuales solicitadas
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
auc_roc_xgb = roc_auc_score(y_test, y_probs_xgb)
# El enunciado pide explícitamente el F2-Score (beta=2)
f2_xgb = fbeta_score(y_test, y_pred_xgb, beta=2)

# 2. Imprimir el bloque de resultados generales
print("======================================================================")
print("                 MÉTRICAS EN CONJUNTO DE TEST (XGBOOST)               ")
print("======================================================================")
print(f"Accuracy General: {accuracy_xgb:.4f}")
print(f"AUC-ROC Score:    {auc_roc_xgb:.4f}")
print(f"F2-Score:         {f2_xgb:.4f}  <-- (Prioriza la detección de abandono)")
print("-" * 70)

# 3. Reporte detallado por clase (Precision, Recall, F1-Score, Support)
print("\nReporte de Clasificación Completo:")
print(classification_report(y_test, y_pred_xgb, target_names=['No Attrition', 'Attrition']))
print("-" * 70)

# 4. Matriz de Confusión Gráfica
print("\nGenerando Matriz de Confusión...")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
disp_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=['No Attrition', 'Attrition'])

# Estilo visual limpio para el informe
fig, ax = plt.subplots(figsize=(6, 5))
disp_xgb.plot(cmap='Greens', ax=ax, values_format='d')
plt.title('Matriz de Confusión - XGBoost (Test)', fontsize=12, pad=10)
plt.grid(False)
plt.show()

In [ ]:
# 1. Extraer el mejor modelo entrenado y el preprocesador del pipeline
mejor_modelo_xgb = xgb_search.best_estimator_
preprocesador = mejor_modelo_xgb.named_steps['preprocesamiento']

# 2. Recuperar los nombres de las variables después de la transformación
# Variables numéricas mantienen su nombre
nombres_num = COLS_NUMERICAS
# Variables categóricas obtienen los nombres generados por OneHotEncoder
nombres_cat = preprocesador.named_transformers_['cat'].get_feature_names_out(COLS_NOMINALES)

# Combinar todas en una lista completa de características
todas_las_features = list(nombres_num) + list(nombres_cat)

# 3. Obtener las importancias del clasificador XGBoost
importancias_xgb = mejor_modelo_xgb.named_steps['clf'].feature_importances_

# 4. Crear un DataFrame para ordenar los datos con facilidad
df_importancia_xgb = pd.DataFrame({
    'Variable': todas_las_features,
    'Importancia': importancias_xgb
}).sort_values(by='Importancia', ascending=False)

# 5. Graficar las 15 variables más importantes
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importancia', 
    y='Variable', 
    data=df_importancia_xgb.head(15), # Mostramos el Top 15 para mantenerlo limpio
    palette='viridis'
)
plt.title('Top 15 - Importancia de Variables en XGBoost (Optimizado)', fontsize=14, pad=15)
plt.xlabel('Importancia (Gini Gain / Weight)', fontsize=12)
plt.ylabel('Variables / Atributos', fontsize=12)
plt.tight_layout()
plt.show()